# Final group project: starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/project_starter.ipynb)

**Machine Learning with Python** · Minor Amsterdam Data Science and AI · Fall 2026

This notebook loads the data of both problems of the project, with English column names, and shows the first steps: for problem 1, the CBS neighborhood statistics and a health table that merges with them; for problem 2, the hotel reviews. Everything after that is up to your group. Read the project description on Canvas first.

**Working together in Colab**

1. One member chooses **File → Save a copy in Drive**, then **Share** (top right) and gives the other members the role *Editor*. Everybody now opens the same notebook.
2. Every member has their own runtime: variables are not shared. Run the cells you need yourself after you open the notebook.
3. Two people typing in the same cell at the same time can overwrite each other. Split the work by section, or use one notebook per part (for example `data.ipynb` and `models.ipynb`), and save often.
4. The files you upload to a Colab session disappear when it ends. Keep your own data files in a shared Google Drive folder, and mount it with `from google.colab import drive; drive.mount("/content/drive")`.

You may also work on your own laptop (see the Laptop lab guide); the data load in the same way.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/data/"

## Problem 1. Neighborhood statistics (CBS, 2024)

The *Kerncijfers wijken en buurten 2024* of Statistics Netherlands (CBS), version of October 2026: one row per region, for the whole country, every municipality, district and neighborhood. We translated the column names into English; the codebook gives the meaning, the unit and the original CBS code of every column. We use the 2024 edition because it is complete: in the 2025 and 2026 editions, income, education and employment are not yet published.

In [ ]:
cbs = pd.read_csv(DATA + "cbs_kwb_2024.csv.gz", low_memory=False)
codebook = pd.read_csv(DATA + "cbs_kwb_2024_codebook.csv")
print(cbs.shape)
print(cbs["region_type"].value_counts())
codebook.head(15)

**Run.** Search the codebook for a word, for example *income*:

In [ ]:
codebook[codebook["description"].str.contains("income", case=False)]

**Run.** Choose one level. Most health data are published per municipality, so start there:

In [ ]:
municipalities = cbs[cbs["region_type"] == "Municipality"].copy()
print(municipalities.shape)
municipalities[["gwb_code", "region_name", "population", "income_per_inhabitant", "pct_poverty", "urbanity"]].head()

**Good to know about these data.**

- **Gaps.** CBS leaves a value empty when it is unknown, unreliable or secret. Small neighborhoods therefore have many more gaps than municipalities: the gaps are not random (Lecture 7, "Why is it missing?").
- **Rounding.** To protect privacy, CBS rounds the counts of districts and neighborhoods at random to multiples of 5.
- **Counts and shares.** Many columns are counts (`population`, `edu_high`, `cars`, ...). Compare regions with shares or rates, for example `edu_high / population`, not with raw counts: a large city has more of everything.
- **Codes.** `gwb_code` is the key to merge on: `GM0363` is the municipality of Amsterdam, `WK036300` a district, `BU03630000` a neighborhood. `boundary_change` tells you whether a region changed since the year before.

### A ready-made health table (RIVM, 2024)

The national health monitor of the GGDs, CBS and RIVM asks adults every few years about their health, lifestyle and living environment. RIVM turns the answers into an estimate for every municipality, district and neighborhood, on the same 2024 division as the CBS table: the codes match one to one. All values are percentages of the adults (18 or older) in the region; the codebook explains every column.

In [ ]:
health = pd.read_csv(DATA + "rivm_health_2024.csv.gz")
health_codebook = pd.read_csv(DATA + "rivm_health_2024_codebook.csv")
print(health.shape)
health_codebook

**Run.** Merge the two tables on the region code. `indicator=True` adds a column `_merge` that tells you, for every row, whether it found a partner:

In [ ]:
merged = municipalities.merge(health.drop(columns=["municipality", "region_type"]),
                              on="gwb_code", how="left", indicator=True)
print(merged["_merge"].value_counts())

merged.plot.scatter(x="pct_poverty", y="lonely", alpha=0.5)
plt.xlabel("persons in poverty (%)")
plt.ylabel("lonely adults (%)")
plt.show()

**Good to know about the health table.**

- **These are estimates, not counts.** RIVM estimates every region with a statistical model from the survey answers and from characteristics of the region, such as age, income and household composition (the same kind of variables as in the CBS table). A model of yours that predicts them well from CBS data partly rediscovers that model. Keep this in mind when you interpret your results, and say it in your report.
- **Gaps.** Small neighborhoods have no estimate (about one neighborhood in five).
- **Country row.** The country has code `NL01` here and `NL00` in the CBS table, so it does not merge; you do not need it.

### More data, in English

You may use the health table, or bring a different outcome. Most Dutch sources are in Dutch; these have English pages or work well with the translate button of your browser. Merge on `gwb_code` (or the municipality code `GM....`), and use data for 2024 where you can.

| Source | What you find there |
|---|---|
| [CBS StatLine, English interface](https://opendata.cbs.nl/statline/#/CBS/en/) | All tables of Statistics Netherlands, for example registered crime, income or energy use per municipality. Choose *Regions* and download as CSV. |
| [RIVM StatLine](https://statline.rivm.nl) | Health tables of RIVM, among them the one above (table 50150NED) for other years and age groups (Dutch). |
| [VZinfo](https://www.vzinfo.nl/) | Regional figures on diseases, mortality and risk factors, per municipality (Dutch). |
| [Atlas Leefomgeving](https://www.atlasleefomgeving.nl/) | The environment: air quality, noise, green space, heat (Dutch). |

Upload your file to the session or to a shared Drive folder, read it with `pd.read_csv`, and check the merge with `indicator=True` as above. Report how many rows did not find a partner, and why.

## Problem 2. Hotel reviews in Europe

515,738 reviews of 1,492 luxury hotels in six European cities (London, Barcelona, Paris, Amsterdam, Vienna, Milan), collected from Booking.com (J. Liu, *515K Hotel Reviews Data in Europe*, Kaggle, 2017, CC0). The file is 46 MB; loading it takes a minute.

In [ ]:
reviews = pd.read_csv(DATA + "hotel_reviews.csv.gz")
print(reviews.shape)
reviews.head(3).T

In [ ]:
print("hotels:", reviews["Hotel_Name"].nunique())
reviews["Reviewer_Score"].plot.hist(bins=30)
plt.xlabel("score given by the reviewer")
plt.show()

**Good to know about these data.**

- **One hotel, many reviews.** If reviews of the same hotel are in both your training and your test set, a model can learn the hotel instead of the review, and its score will be too optimistic. Decide what you predict for, new reviews or new hotels, and split accordingly, for example with `GroupKFold(n_splits=5)` and `groups=reviews["Hotel_Name"]`.
- **Text.** `Positive_Review` and `Negative_Review` are free text; "No Negative" and "No Positive" mean the reviewer left the field empty. The word counts are ready-made features.
- **Tags** are stored as one string that looks like a Python list, for example `"[' Leisure trip ', ' Couple ', ...]"`.
- **Location.** `lat` and `lng` are missing for 17 hotels (3,268 reviews); the address is always there.
- **Size.** For a first exploration, work on a sample: `reviews.sample(50_000, random_state=0)`.

## Your project

Delete the problem you do not use, and continue below. A structure that keeps the work of three people in order:

1. **Question** (in a text cell): what do you want to find out, and how will you measure success?
2. **Data**: loading, merging, cleaning, and what you lost on the way.
3. **Split**: the test set, before you look at the results of any model.
4. **Models**: a baseline and at least two models, each in a pipeline, tuned by cross-validation.
5. **Analysis**: one analysis beyond the scores.
6. **Test set, once**, and the figures and tables for the report.

Before you hand in: **Runtime → Restart session and run all**, and check that every number in your report comes out of this notebook.